In [1]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.sandbox.regression.gmm import IV2SLS
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

oilp = pd.read_csv('data/wti_spot_oil.csv')
oilp = oilp.rename(columns={'WTISPLC': 'price'})

oilp['DATE'] = pd.to_datetime(oilp['DATE']) - pd.offsets.MonthEnd(1)
oilp['pc_1q'] = np.log(oilp['price'].shift(1) / oilp['price'].shift(2))
oilp['pc_2q'] = np.log(oilp['price'].shift(2) / oilp['price'].shift(3))

oilp.to_csv('cleaned_data/oil_prices.csv')

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')

###############################################
                ## ESTIMATION ##
###############################################

### OLS ###
def OLS(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=None))
    return regs


### PLD OLS ###
def OLS_PLD(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        regs.append(sm.OLS(errors, revisions).fit(cov_type='cluster', cov_kwds = {"groups":df['ID']}))
    return regs


### ID FE ###
def ID_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        regs.append(sm.OLS(y, x).fit(cov_type='cluster', cov_kwds = {"groups":df['ID']}))
    return regs


### AR Estimates ###
def AR(df, end_date):
    df = df.loc[(df['DATE'] >= '1965-06-30') & (df['DATE'] <= end_date)]  # Filter data
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    return reg


### Parameter Estimates ###
def params(ols, pldols, fe, ar1):
    params = []
    params.append(ols / (1 + ols))
    params.append(1 / (1 + ols))
    params.append((-((2 * pldols) + 1) + np.sqrt(((2 * pldols) + 1)**2 - 4 * (pldols + (pldols * ar1**2) + 1) * pldols)) / (2 * (pldols + (pldols * ar1**2) + 1)))
    params.append((-((2 * fe) + 1) + np.sqrt(((2 * fe) + 1)**2 - 4 * (fe + (fe * ar1**2) + 1) * fe)) / (2 * (fe + (fe * ar1**2) + 1)))
    return params


def compute_regs(date, mean, ind):
    mean_regs = OLS(mean, f'{date}')
    ind_regs = OLS_PLD(ind, f'{date}')
    ind_regs_fe = ID_FE(ind, f'{date}')
    ar_1 = AR(vintage_trim, f'{date}')
    regs = [mean_regs, ind_regs, ind_regs_fe, ar_1]
    parameters = params(mean_regs[3].params[1], ind_regs[3].params[1], ind_regs_fe[3].params[1], ar_1.roots)
    return regs, parameters

regs_2020, parameters_2020 = compute_regs('2020-06-30', mean_spf_trim, ind_spf_trim)
%store regs_2020
%store parameters_2020

regs_2022, parameters_2022 = compute_regs('2022-12-31', mean_spf_trim, ind_spf_trim)
%store regs_2022
%store parameters_2022

/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2020' (list)
Stored 'parameters_2020' (list)
Stored 'regs_2022' (list)
Stored 'parameters_2022' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [8]:
## OIL INSTRUMENT ##
def IV_OIL_OLS(df, oil, end_date):
    df = df.loc[(df['DATE'] >= '1986-09-30') & (df['DATE'] <= end_date)]
    oil = oil.loc[(oil['DATE'] >= '1986-09-30') & (oil['DATE'] <= end_date)]
    revisions = df[f'r_t3']
    x = sm.add_constant(revisions)
    errors = df[f'e_t3']
    L1_oil = oil['pc_1q']
    L2_oil = oil['pc_2q']
    w = np.column_stack((L1_oil, L2_oil))
    w = sm.add_constant(w)
    initial = IV2SLS(errors, x, instrument = w).fit()
    reg = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return reg

date = '2020-06-30'
IV_OIL_OLS(mean_spf_trim, oilp, date).summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                          IV2SLS Regression Results                           
==============================================================================
Dep. Variable:                   e_t3   R-squared:                      -0.082
Model:                         IV2SLS   Adj. R-squared:                 -0.091
Method:                     Two Stage   F-statistic:                     1.255
                        Least Squares   Prob (F-statistic):              0.265
Date:                Thu, 09 May 2024                                         
Time:                        09:07:47                                         
No. Observations:                 136                                         
Df Residuals:                     134                                         
Df Model:                           1                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0006      0.001     -0.472      0.638      -0.003       0.002
r_t3           0.6874      0.614      1.120      0.265      -0.526       1.901
==============================================================================
Omnibus:                       17.459   Durbin-Watson:                   0.837
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               43.174
Skew:                          -0.448   Prob(JB):                     4.22e-10
Kurtosis:                       5.611   Cond. No.                         267.
==============================================================================
"""

In [10]:
regs_2022[3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            AutoReg Model Results                             
==============================================================================
Dep. Variable:                     t3   No. Observations:                  231
Model:                     AutoReg(1)   Log Likelihood                 788.985
Method:               Conditional MLE   S.D. of innovations              0.008
Date:                Thu, 09 May 2024   AIC                          -1571.970
Time:                        10:38:15   BIC                          -1561.656
Sample:                             1   HQIC                         -1567.810
                                  231                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0016      0.001      1.817      0.069      -0.000       0.003
t3.L1          0.9607      0.018     52.915      0.000       0.925       0.996
                                    Roots                                    
=============================================================================
                  Real          Imaginary           Modulus         Frequency
-----------------------------------------------------------------------------
AR.1            1.0409           +0.0000j            1.0409            0.0000
-----------------------------------------------------------------------------
"""

In [6]:
parameters_2020[3]

array([0.5570091])